In [ ]:
!pip install transformers datasets accelerate sentencepiece

In [ ]:
# ==========================================
# IMPORTS
# ==========================================

import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

In [ ]:
# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "../data/processed/cleaned_data.csv"
)

df.head()

In [ ]:
# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(

    df['clean_text'],

    df['label_encoded'],

    test_size=0.2,

    random_state=42,

    stratify=df['label_encoded']
)

In [ ]:
# ==========================================
# MODEL NAME
# ==========================================

model_name = "xlm-roberta-base"

In [ ]:
# ==========================================
# TOKENIZER
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

train_encodings = tokenizer(

    list(X_train),

    truncation=True,

    padding=True,

    max_length=128
)

test_encodings = tokenizer(

    list(X_test),

    truncation=True,

    padding=True,

    max_length=128
)

In [ ]:
# ==========================================
# DATASET CLASS
# ==========================================

class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {

            key: torch.tensor(val[idx])

            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)   # Fix: convert to list

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# ==========================================
# LOAD MODEL
# ==========================================

model = AutoModelForSequenceClassification.from_pretrained(

    model_name,

    num_labels=3
)

In [ ]:
def compute_metrics(pred):
    predictions = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids
    return {
        "accuracy":  accuracy_score(labels, predictions),
        "f1":        f1_score(labels, predictions, average="weighted"),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall":    recall_score(labels, predictions, average="weighted", zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="../results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,          
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="../logs",
    no_cuda=True,                
    report_to="none"             
)

In [ ]:
# ==========================================
# TRAINER
# ==========================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
# ==========================================
# TRAIN MODEL
# ==========================================

trainer.train()

In [ ]:
# ==========================================
# EVALUATE
# ==========================================

results = trainer.evaluate()

results

In [ ]:
# ==========================================
# SAVE RESULTS
# ==========================================

results_df = pd.DataFrame([results])

results_df.to_csv(

    "../results/transformer_results.csv",

    index=False
)

print("Transformer results saved.")

In [ ]:
# ==========================================
# SAVE MODEL
# ==========================================

model.save_pretrained(
    "../models/transformers/xlm_roberta"
)

tokenizer.save_pretrained(
    "../models/transformers/xlm_roberta"
)